# Modelo v3 — detector de huevos Roto / Sano (YOLOv8)

**Qué lo hace distinto de los detectores de huevos que ya existen**

1. **Distingue marcas de tinta de fisuras reales.** Los huevos comerciales vienen sellados (fecha,
   código de granja) y en campo se les raya o ensucia. El modelo 1 bota el **56 %** de los huevos
   sanos solo por una raya de lapicero. Aquí se entrena con **negativos duros sintéticos**: sellos,
   rayas y suciedad dibujados sobre huevos *sanos*, etiquetados como sanos. Se mide antes y después.
2. **Robustez medida a la foto real**: desenfoque, cámara movida, sobre y subexposición y JPEG del
   celular, con aumentaciones apuntadas a las fallas que se midieron en campo.
3. **Desempeño por color de cáscara** (blanco / marrón): una grieta contrasta distinto en cada una.
   El color se estima del recorte y se valida contra fuentes que traen el color real.
4. **Curación multi-fuente auditada**: 9 datasets públicos combinados con detección automática de su
   convención de etiquetado, deduplicación entre fuentes (muchos proyectos son copias de otros) y un
   **benchmark común con el modelo 1** que nunca se toca en el entrenamiento.

**Split: 80/10/10** si tras deduplicar quedan ≥ 3.000 originales (lo esperado: ~4.000+); si no,
**70/15/15**. La regla está en la celda 5 y el número real se imprime.

**Modelo: YOLOv8l a 640**, entrenamiento largo: mínimo 150 épocas; después, si pasan 30 épocas sin una
mejora significativa de mAP50-95, **no corta en seco**: entra en una fase final de 20 épocas sin mosaico
y con la tasa de aprendizaje bajando a su mínimo, y termina. Tope 300 épocas.

## Antes de empezar
1. **Entorno de ejecución → Cambiar tipo de entorno → GPU** (T4 sirve; A100/L4 si tienes Colab Pro).
2. Secreto `ROBOFLOW_API_KEY` con tu **Private API Key** (panel 🔑, con *Acceso al notebook*).
3. Sube **`huevos.zip`** (tu primer dataset, está en `D:\eggdetection\huevos.zip`) a
   **`MyDrive/eggdetection/huevos.zip`**. Su test es el benchmark común con el modelo 1.
4. **Ejecutar todas.** Pide el permiso de Drive al inicio. **Es largo.** Referencia medida: tu modelo 1
   (YOLOv8m, 678 fotos) tardó 23 s/época en Colab. La v3 tiene ~6–7 veces más fotos de train y YOLOv8l
   cuesta ~2 veces más → **~5 min/época en una T4: de 14 a 25 horas** (L4 ≈ 2× más rápida, A100 ≈ 4–5×).
   Desde la época 2, la celda 11 imprime el ritmo **real** y cuánto falta. Si Colab se desconecta,
   vuelve a correr 1–10 y reanuda en la celda 11 con `REANUDAR`: todo se guarda en Drive época a época.

In [ ]:
# @title 1. Entorno (mismas versiones probadas localmente con un entrenamiento real)
import subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "SIN GPU -> Entorno de ejecucion > Cambiar tipo de entorno > GPU")
assert gpu, "Activa la GPU antes de seguir"
%pip install -q ultralytics==8.4.157 roboflow==1.5.0 albumentations==2.0.8 onnx onnxruntime onnxslim
# roboflow 1.5.0 pide numpy<2.4: si pip lo cambio con numpy ya cargado, seguir mezclaria dos versiones.
import sys
nuevo = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"], capture_output=True, text=True).stdout.strip()
if "numpy" in sys.modules and sys.modules["numpy"].__version__ != nuevo:
    raise RuntimeError(f"pip cambio numpy {sys.modules['numpy'].__version__} -> {nuevo}. Entorno de ejecucion > "
                       "Reiniciar sesion y vuelve a 'Ejecutar todas' (la segunda vez la instalacion no cambia nada).")
import ultralytics
ultralytics.checks()

In [ ]:
# @title 2. Configuración
from pathlib import Path
from datetime import datetime

# ---------------------------------------------------------------- fuentes
# mapa: clase de la fuente -> "Roto" | "Sano" | "EXCLUIR" (es un huevo que no se puede mapear:
#       la imagen entera sale) | "IGNORAR" (no es un huevo: se borra solo la caja).
# convencion: "auto" la detecta la auditoria (celda 4); "huevo"/"region" la fuerzan.
# color: clase -> color real de cascara (solo algunas fuentes lo traen; sirven para calibrar).
FUENTES = [
    dict(nombre="huevos", local="/content/drive/MyDrive/eggdetection/huevos.zip",
         mapa={"Roto": "Roto", "Sano": "Sano"}, convencion="huevo"),
    dict(nombre="indra", ws="indra-bayu", proyecto="egg-shell-cracked-detections", version=1,
         mapa={"egg": "Sano", "cracked": "Roto", "0": "EXCLUIR"}, convencion="auto"),
    dict(nombre="360a", ws="360digitmg-swnfx", proyecto="egg-crack-detection-yjhsy", version=1,
         mapa={"Cracked Egg Brown": "Roto", "Cracked Egg White": "Roto", "Normal Egg Brown": "Sano", "Normal Egg White": "Sano"},
         color={"Cracked Egg Brown": "marron", "Normal Egg Brown": "marron", "Cracked Egg White": "blanco", "Normal Egg White": "blanco"},
         convencion="auto"),
    dict(nombre="360b", ws="digit360mg", proyecto="egg-crack-detection-fahfj", version=1,
         mapa={"Brown cracked egg": "Roto", "White cracked egg": "Roto", "Brown egg": "Sano", "White egg": "Sano"},
         color={"Brown cracked egg": "marron", "Brown egg": "marron", "White cracked egg": "blanco", "White egg": "blanco"},
         convencion="auto"),
    dict(nombre="jqutx", ws="new-workspace-jqutx", proyecto="egg-crack-detection-ek7qj", version=3,
         mapa={"crack": "Roto", "normal": "Sano"}, convencion="auto"),
    dict(nombre="carlos", ws="carlos-oliveira-5vnxi", proyecto="crack-egg-8wlgl", version=2,
         mapa={"crack": "Roto", "good": "Sano"}, convencion="auto"),
    dict(nombre="aau", ws="aau-nupnz", proyecto="egg-crack-9lbqk", version=1,
         mapa={"crack": "Roto", "egg": "Sano"}, convencion="auto"),
    dict(nombre="cracked", ws="crackedeggs", proyecto="cracked-eggs", version=1,
         mapa={"cracked-egg": "Roto", "normal-egg": "Sano"}, convencion="auto"),
    dict(nombre="ducks", ws="ducks-sfodd", proyecto="egg-quality-o7xyt", version=7,   # segmentacion -> cajas
         mapa={"egg-clean": "Sano", "egg-cracked": "Roto", "egg-stained": "Sano",        # manchado = negativo duro real
               "clean": "EXCLUIR", "duck-clean": "EXCLUIR", "duck-cracked": "EXCLUIR", "duck-stained": "EXCLUIR",
               "reject": "EXCLUIR"}, convencion="auto"),
]
PRIORIDAD = [f["nombre"] for f in FUENTES]     # ante la MISMA foto en dos fuentes, gana la primera

# ---------------------------------------------------------------- modelo
# yolov8l: ~5 min/epoca en T4 (14-25 h). yolov8m: ~2,4 min/epoca (7-12 h), algo menos preciso y la
# mitad de latencia en el servidor (~0,6 s por foto en el t2.large contra ~1,3 s).
MODELO_BASE = "yolov8l.pt"  # @param ["yolov8m.pt", "yolov8l.pt", "yolov8x.pt"]
IMGSZ = 640
BATCH = {"yolov8m.pt": 32, "yolov8l.pt": 16, "yolov8x.pt": 8}[MODELO_BASE]   # T4 16 GB con AMP

# Regla de las sesiones anteriores + cierre ordenado (celda 11)
MIN_EPOCAS     = 150    # @param {type:"integer"}
PACIENCIA      = 30     # @param {type:"integer"}
MEJORA_MINIMA  = 0.003  # @param {type:"number"}   # 0.3 puntos de mAP50-95
EPOCAS_FINALES = 20     # @param {type:"integer"}  # fase final sin mosaico, LR al minimo
EPOCAS_MAX     = 300    # @param {type:"integer"}

# ---------------------------------------------------------------- datos
FRACCION_NEGATIVOS = 0.35   # fraccion de imagenes de train con huevos sanos que reciben una copia marcada
SEMILLA = 42
COSTO_FN, COSTO_FP = 12.0, 1.0          # dejar pasar un roto cuesta 12 veces botar uno sano
MAX_REVISION_SANOS = 0.05                # banda de revision: <=5 % de los sanos de val
CONF_MIN, IOU_NMS, NITIDEZ_MIN = 0.15, 0.45, 4.0   # mismos del backend

TRABAJO = Path("/content/v3"); FUENTES_DIR = TRABAJO / "fuentes"; DATASET = TRABAJO / "dataset"
from google.colab import drive
drive.mount("/content/drive")
CARPETA_RUNS = Path("/content/drive/MyDrive/eggdetection/runs_v3"); CARPETA_RUNS.mkdir(parents=True, exist_ok=True)
NOMBRE_RUN = f"v3_{Path(MODELO_BASE).stem}_{datetime.now():%Y%m%d_%H%M}"
print("run:", NOMBRE_RUN, "->", CARPETA_RUNS / NOMBRE_RUN)

### Módulo `v3_datos_lib.py` — curación multi-fuente, auditoría, dedup, split y negativos duros
Archivo probado localmente; se escribe tal cual.

In [ ]:
%%writefile v3_datos_lib.py
# =============================================================================
#  v3 — Curacion multi-fuente para DETECCION de huevos (Roto / Sano)
#  (probado localmente con v3/test_v3_datos_lib.py antes de ir a Colab)
#
#  Lo que hace, y por que:
#   1. Lee exports YOLO de Roboflow (deteccion o segmentacion -> caja).
#   2. AUDITA cada fuente y detecta su convencion de etiquetado:
#        "huevo"  -> la caja de roto rodea el huevo entero   (lo que usa el backend)
#        "region" -> la caja de roto rodea SOLO la grieta, dentro de una caja de huevo
#      Mezclar ambas sin convertir le ensena al modelo dos cosas incompatibles.
#   3. Mapea las clases de cada fuente a {Roto, Sano}. Una clase que es un huevo
#      pero no se puede mapear EXCLUYE la imagen: si solo se borrara la caja, ese
#      huevo quedaria como "fondo" y el modelo aprenderia que un huevo no es huevo.
#   4. Deduplica ENTRE fuentes (muchos proyectos de Roboflow son copias de otros),
#      con hash perceptual invariante a rotaciones de 90 grados y espejos.
#   5. Agrupa "el mismo huevo" (copias, videos, rafagas, casi-duplicados con tope)
#      y reparte GRUPOS estratificando por (fuente, tiene_roto).
#   6. Reserva el test del dataset original como BENCHMARK comun con el modelo 1,
#      junto con todo lo que se le parezca en otras fuentes: sin eso la comparacion
#      v3 vs modelo 1 estaria contaminada.
#   7. Negativos duros sinteticos (el diferencial): sellos de tinta, rayas de
#      lapicero y suciedad dibujados sobre huevos SANOS de train. El modelo 1
#      bota el 56 % de los huevos sanos con una raya de lapicero.
#   8. Color de cascara (blanco / marron) estimado del recorte, calibrado contra
#      las fuentes que traen el color en la etiqueta.
# =============================================================================
from __future__ import annotations

import math
import random
import re
import shutil
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

EXT_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CARPETAS_SPLIT = {"train": "train", "valid": "val", "val": "val", "validation": "val", "test": "test"}
CLASES = ["Roto", "Sano"]               # MISMO orden que el backend: 0=Roto, 1=Sano
IGNORAR, EXCLUIR = "IGNORAR", "EXCLUIR"  # no es huevo -> se borra la caja | es huevo sin mapeo -> fuera la imagen


def norm(nombre: str) -> str:
    """'Cracked_Egg-Brown ' -> 'cracked egg brown': los nombres de clase de Roboflow varian."""
    return re.sub(r"[\s_\-]+", " ", str(nombre).strip().lower())


def normalizar_mapa(mapa: dict | None) -> dict:
    return {norm(k): v for k, v in (mapa or {}).items()}


# ------------------------------------------------------------------ nombres
def nombre_base(nombre: str) -> str:
    """'foto_jpg.rf.ab12.jpg' -> 'foto'. Las copias aumentadas comparten la base."""
    base = nombre.split(".rf.")[0]
    base = re.sub(r"\.(jpe?g|png|bmp|webp)$", "", base, flags=re.I)
    return re.sub(r"_(jpe?g|png|bmp|webp)$", "", base, flags=re.I)


RE_VIDEO = re.compile(r"^(\d{8}_\d{6})_(\d{1,5})$")
RE_FECHA = re.compile(r"^(\d{4})-(\d{2})-(\d{2})-(\d{2})_(\d{2})_(\d{2})(?:_(\d+))?$")
RE_FRAME = re.compile(r"^(.+?)[_\-\s]*frame[_\-\s]*\d+$", re.I)


def clave_video(base: str) -> str | None:
    if m := RE_VIDEO.match(base):
        return "video:" + m.group(1)
    if m := RE_FRAME.match(base):
        return "video:" + m.group(1).lower()
    return None


def fecha_de(base: str) -> datetime | None:
    if m := RE_FECHA.match(base):
        y, mo, d, h, mi, s, micro = m.groups()
        return datetime(int(y), int(mo), int(d), int(h), int(mi), int(s), int((micro or "0")[:6].ljust(6, "0")))
    return None


# ------------------------------------------------------------ lectura YOLO
def leer_nombres(data_yaml: Path) -> list[str]:
    """names de un data.yaml de Roboflow (lista o dict), sin depender de PyYAML."""
    txt = data_yaml.read_text(encoding="utf-8", errors="ignore")
    try:
        import yaml
        d = yaml.safe_load(txt)
        n = d.get("names")
        return [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)
    except Exception:
        m = re.search(r"names:\s*\[(.*?)\]", txt, re.S)
        if m:
            return [x.strip().strip("'\"") for x in m.group(1).split(",") if x.strip()]
        return re.findall(r"^\s*\d+:\s*['\"]?(.+?)['\"]?\s*$", txt.split("names:", 1)[-1], re.M)


def leer_etiquetas(ruta_txt: Path, nombres: list[str]) -> list[tuple[str, float, float, float, float]]:
    """Cajas normalizadas (clase, cx, cy, w, h). Poligonos de segmentacion -> caja envolvente."""
    cajas = []
    if not ruta_txt.exists():
        return cajas
    for ln in ruta_txt.read_text(encoding="utf-8", errors="ignore").splitlines():
        p = ln.split()
        if len(p) < 5:
            continue
        c = int(float(p[0]))
        if c >= len(nombres):
            continue
        v = [float(x) for x in p[1:]]
        if len(v) == 4:
            cx, cy, w, h = v
        else:                                      # poligono x1 y1 x2 y2 ...
            xs, ys = v[0::2], v[1::2]
            x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
            cx, cy, w, h = (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1
        if w > 0 and h > 0:
            cajas.append((nombres[c], cx, cy, w, h))
    return cajas


def inventariar(raiz: Path, fuente: str) -> list[dict]:
    """Export YOLO de Roboflow: <raiz>/<split>/images + labels + data.yaml."""
    yml = next(iter(sorted(raiz.rglob("data.yaml"))), None)
    if yml is None:
        raise FileNotFoundError(f"{fuente}: no hay data.yaml en {raiz}")
    nombres = leer_nombres(yml)
    items = []
    for carpeta in sorted(p for p in yml.parent.iterdir() if p.is_dir()):
        split = CARPETAS_SPLIT.get(carpeta.name.lower())
        if split is None or not (carpeta / "images").is_dir():
            continue
        for img in sorted((carpeta / "images").iterdir()):
            if img.suffix.lower() in EXT_IMG:
                items.append({"ruta": img, "fuente": fuente, "split_orig": split,
                              "base": f"{fuente}:{nombre_base(img.name)}", "base_corta": nombre_base(img.name),
                              "cajas_src": leer_etiquetas(carpeta / "labels" / (img.stem + ".txt"), nombres)})
    return items


# ------------------------------------------------------------------ auditoria
def _xyxy(c):
    _, cx, cy, w, h = c
    return (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)


def _ioa(a, b) -> float:
    """Fraccion del area de a que cae dentro de b."""
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    return inter / max(1e-9, (a[2] - a[0]) * (a[3] - a[1]))


def auditar(items: list[dict], mapa: dict[str, str]) -> dict:
    """Estadisticas por clase y convencion detectada ('huevo' o 'region')."""
    por_clase = defaultdict(list)
    for it in items:
        for c in it["cajas_src"]:
            por_clase[c[0]].append(c[3] * c[4])
    resumen = {k: {"cajas": len(v), "area_mediana": float(np.median(v))} for k, v in sorted(por_clase.items())}

    mapa = normalizar_mapa(mapa)
    roto_src = {k for k, v in mapa.items() if v == "Roto"}
    sano_src = {k for k, v in mapa.items() if v == "Sano"}
    ioas, razones = [], []
    for it in items:
        sanos = [_xyxy(c) for c in it["cajas_src"] if norm(c[0]) in sano_src]
        if not sanos:
            continue
        for c in it["cajas_src"]:
            if norm(c[0]) in roto_src:
                a = _xyxy(c)
                j = max(range(len(sanos)), key=lambda k: _ioa(a, sanos[k]))
                ioas.append(_ioa(a, sanos[j]))
                s = sanos[j]
                razones.append((c[3] * c[4]) / max(1e-9, (s[2] - s[0]) * (s[3] - s[1])))
    region = bool(ioas) and np.median(ioas) >= 0.8 and np.median(razones) <= 0.5
    return {"clases": resumen, "convencion": "region" if region else "huevo",
            "ioa_roto_en_sano": float(np.median(ioas)) if ioas else None,
            "area_roto_sobre_sano": float(np.median(razones)) if razones else None,
            "sin_clase_sano": not sano_src, "imagenes": len(items),
            "originales": len({it["base"] for it in items})}


def mapear(items: list[dict], mapa: dict[str, str], convencion: str, color_por_clase: dict | None = None) -> dict:
    """Convierte cajas de la fuente a (Roto|Sano). Devuelve contadores de lo descartado."""
    stats = Counter()
    mapa, color_por_clase = normalizar_mapa(mapa), normalizar_mapa(color_por_clase)
    for it in items:
        cajas, excluir, sin_mapa = [], False, set()
        crudas = [(norm(c[0]),) + tuple(c[1:]) for c in it["cajas_src"]]
        if convencion == "region":
            huevos = [c for c in crudas if mapa.get(c[0]) == "Sano"]
            grietas = [c for c in crudas if mapa.get(c[0]) == "Roto"]
            for g in grietas:
                if not any(_ioa(_xyxy(g), _xyxy(h)) >= 0.5 for h in huevos):
                    excluir = True                 # grieta sin huevo que la contenga: ambiguo
            for h in huevos:
                roto = any(_ioa(_xyxy(g), _xyxy(h)) >= 0.5 for g in grietas)
                cajas.append(("Roto" if roto else "Sano", h[1], h[2], h[3], h[4], (color_por_clase or {}).get(h[0])))
            crudas = [c for c in crudas if mapa.get(c[0]) not in ("Roto", "Sano")]
        for c in crudas:
            destino = mapa.get(c[0], EXCLUIR)
            if c[0] not in mapa:
                sin_mapa.add(c[0])
            if destino == EXCLUIR:
                excluir = True
            elif destino in ("Roto", "Sano"):
                cajas.append((destino, c[1], c[2], c[3], c[4], (color_por_clase or {}).get(c[0])))
        it["cajas"], it["excluir"] = cajas, excluir
        stats["imagenes_excluidas"] += excluir
        stats["imagenes_sin_cajas"] += (not excluir and not cajas)
        for k in sin_mapa:
            stats[f"clase sin mapeo: {k}"] += 1
    return dict(stats)


# ------------------------------------------------------------ hash perceptual
def _dhash_de(im: Image.Image, lado: int = 16) -> np.ndarray:
    a = np.asarray(im.resize((lado + 1, lado), Image.BILINEAR), dtype=np.int16)
    return np.packbits((a[:, 1:] > a[:, :-1]).flatten()).view(np.uint64).copy()


def hashes_variantes(ruta: Path) -> np.ndarray:
    """8 variantes (4 rotaciones x espejo): (8, 4) uint64. Una sola decodificacion."""
    im = Image.open(ruta).convert("L")
    out = []
    for espejo in (False, True):
        base = im.transpose(Image.FLIP_LEFT_RIGHT) if espejo else im
        for rot in (None, Image.ROTATE_90, Image.ROTATE_180, Image.ROTATE_270):
            out.append(_dhash_de(base.transpose(rot) if rot is not None else base))
    return np.stack(out)


def calcular_hashes(items: list[dict], log=print) -> None:
    log(f"  hash perceptual (256 bits x 8 variantes) de {len(items)} imagenes...")
    for it in items:
        it["hv"] = hashes_variantes(it["ruta"])


def _popcount(x: np.ndarray) -> np.ndarray:
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).sum(axis=-1, dtype=np.int32)
    lut = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
    return lut[x.view(np.uint8)].sum(axis=-1, dtype=np.int32)


def pares_cercanos(items: list[dict], umbral: int) -> list[tuple[int, int, int]]:
    """(distancia, i, j) con distancia invariante a rotacion/espejo <= umbral."""
    H0 = np.stack([it["hv"][0] for it in items])            # (n, 4) variante canonica
    V = np.stack([it["hv"] for it in items])                 # (n, 8, 4)
    pares = []
    for a in range(len(items) - 1):
        d = _popcount(H0[a][None, None, :] ^ V[a + 1:]).min(axis=1)   # vs las 8 variantes del resto
        for b in np.nonzero(d <= umbral)[0]:
            pares.append((int(d[b]), a, a + 1 + int(b)))
    pares.sort()
    return pares


def deduplicar_entre_fuentes(items: list[dict], prioridad: list[str], umbral: int = 4, log=print) -> list[dict]:
    """La MISMA foto en dos fuentes -> se queda la de mayor prioridad (primera en la lista)."""
    rango = {f: i for i, f in enumerate(prioridad)}
    fuera = set()
    for d, i, j in pares_cercanos(items, umbral):
        a, b = items[i], items[j]
        if a["fuente"] == b["fuente"] or i in fuera or j in fuera:
            continue
        perdedor = j if rango.get(a["fuente"], 99) <= rango.get(b["fuente"], 99) else i
        fuera.add(perdedor)
    quitados = Counter(items[k]["fuente"] for k in fuera)
    log(f"  duplicados entre fuentes eliminados: {len(fuera)}  {dict(quitados)}")
    return [it for k, it in enumerate(items) if k not in fuera]


# ------------------------------------------------------------------ grupos
class UnionFind:
    def __init__(self, n: int) -> None:
        self.p, self.tam = list(range(n)), [1] * n
        self.causas: list[Counter] = [Counter() for _ in range(n)]

    def find(self, x: int) -> int:
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a: int, b: int, causa: str) -> bool:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.tam[ra] < self.tam[rb]:
            ra, rb = rb, ra
        self.p[rb] = ra
        self.tam[ra] += self.tam[rb]
        self.causas[ra] += self.causas[rb]
        self.causas[ra][causa] += 1
        return True


def agrupar(items: list[dict], gap_rafaga_s: float = 20.0, duracion_max_rafaga_s: float = 120.0,
            umbral_hash: int = 16, max_grupo_hash: int = 30) -> dict:
    """'Mismo huevo': copia aumentada, video, rafaga (por fuente) y casi-duplicado visual con tope."""
    n = len(items)
    uf, razones = UnionFind(n), Counter()

    def unir(i, j, causa):
        if uf.union(i, j, causa):
            razones[causa] += 1

    for clave, regla in ((lambda it: it["base"], "copia aumentada"),
                         (lambda it: f"{it['fuente']}|{clave_video(it['base_corta'])}"
                          if clave_video(it["base_corta"]) else None, "frame del mismo video")):
        idx = defaultdict(list)
        for i, it in enumerate(items):
            k = clave(it)
            if k:
                idx[k].append(i)
        for grupo in idx.values():
            for j in grupo[1:]:
                unir(grupo[0], j, regla)

    por_fuente = defaultdict(list)
    for i, it in enumerate(items):
        t = fecha_de(it["base_corta"])
        if t:
            por_fuente[it["fuente"]].append((t, i))
    for lista in por_fuente.values():
        lista.sort()
        inicio = lista[0][0]
        for (t0, i0), (t1, i1) in zip(lista, lista[1:]):
            if (t1 - t0).total_seconds() <= gap_rafaga_s and (t1 - inicio).total_seconds() <= duracion_max_rafaga_s:
                unir(i0, i1, "rafaga")
            else:
                inicio = t1

    rechazadas = 0
    for _, i, j in pares_cercanos(items, umbral_hash):
        if uf.find(i) == uf.find(j):
            continue
        if uf.tam[uf.find(i)] + uf.tam[uf.find(j)] > max_grupo_hash:
            rechazadas += 1
            continue
        unir(i, j, "casi-duplicado visual")

    for i, it in enumerate(items):
        it["grupo"] = uf.find(i)
    return {"razones": razones, "n_grupos": len({it["grupo"] for it in items}),
            "rechazadas_por_tope": rechazadas,
            "causas_por_grupo": {r: uf.causas[r] for r in {it["grupo"] for it in items}}}


# ------------------------------------------------------------------ split
def clave_estrato(it: dict) -> str:
    return f"{it['fuente']}|{'con_roto' if any(c[0] == 'Roto' for c in it['cajas']) else 'solo_sano'}"


def repartir(items: list[dict], proporciones: dict[str, float], benchmark: callable | None = None,
             semilla: int = 42) -> dict[int, str]:
    """Grupos -> split. Todo grupo que toque el benchmark va entero al benchmark."""
    grupos_bench = {it["grupo"] for it in items if benchmark and benchmark(it)}
    comp = defaultdict(Counter)
    vistos = set()
    for it in items:
        if it["grupo"] in grupos_bench or (it["grupo"], it["base"]) in vistos:
            continue
        vistos.add((it["grupo"], it["base"]))
        comp[it["grupo"]][clave_estrato(it)] += 1
    total = Counter()
    for c in comp.values():
        total.update(c)
    rng = random.Random(semilla)
    orden = list(comp)
    rng.shuffle(orden)
    orden.sort(key=lambda g: -sum(comp[g].values()))
    cuenta = {s: Counter() for s in proporciones}
    asignado = {g: "benchmark" for g in grupos_bench}
    for g in orden:
        s = max(proporciones, key=lambda s: sum(n * (proporciones[s] - cuenta[s][k] / max(1, total[k]))
                                                 for k, n in comp[g].items()))
        asignado[g] = s
        cuenta[s].update(comp[g])
    return asignado


def revisar_reparto(items: list[dict], asignado: dict[int, str], proporciones: dict[str, float],
                    minimo_relativo: float = 0.6) -> dict[str, float]:
    """Se niega a seguir si un split quedo muy por debajo de su proporcion."""
    por = defaultdict(set)
    for it in items:
        if asignado[it["grupo"]] != "benchmark":
            por[asignado[it["grupo"]]].add(it["base"])
    tot = sum(len(v) for v in por.values())
    real = {s: len(por.get(s, ())) / max(1, tot) for s in proporciones}
    malos = [s for s in proporciones if real[s] < minimo_relativo * proporciones[s]]
    if malos:
        raise RuntimeError("Reparto desbalanceado: " + ", ".join(
            f"{s}={100 * real[s]:.1f}% (objetivo {100 * proporciones[s]:.0f}%)" for s in proporciones))
    return real


def escribir_etiqueta(ruta: Path, cajas: list) -> None:
    ruta.write_text("".join(f"{CLASES.index(c[0])} {c[1]:.6f} {c[2]:.6f} {c[3]:.6f} {c[4]:.6f}\n" for c in cajas))


def materializar(items: list[dict], asignado: dict[int, str], destino: Path) -> dict[str, list[dict]]:
    """UNA imagen por original en todos los splits (prefiere la que venia de valid/test, sin
    aumentacion horneada): Ultralytics aumenta en linea, distinto en cada epoca."""
    if destino.exists():
        shutil.rmtree(destino)
    por_split = defaultdict(dict)
    for it in sorted(items, key=lambda x: (x["split_orig"] == "train", x["ruta"].name)):
        por_split[asignado[it["grupo"]]].setdefault(it["base"], it)
    final = {}
    for split, d in por_split.items():
        (destino / split / "images").mkdir(parents=True, exist_ok=True)
        (destino / split / "labels").mkdir(parents=True, exist_ok=True)
        final[split] = []
        for it in d.values():
            nombre = f"{it['fuente']}__{it['ruta'].stem}"[:180]
            dst = destino / split / "images" / (nombre + it["ruta"].suffix.lower())
            shutil.copy2(it["ruta"], dst)
            escribir_etiqueta(destino / split / "labels" / (nombre + ".txt"), it["cajas"])
            final[split].append({**it, "ruta_final": dst})
    return final


def escribir_data_yaml(destino: Path, splits: list[str]) -> Path:
    y = destino / "data.yaml"
    lineas = [f"path: {destino.as_posix()}", "train: train/images", "val: val/images"]
    for s in splits:
        if s not in ("train", "val"):
            lineas.append(f"{s}: {s}/images")
    lineas += ["nc: 2", "names: ['Roto', 'Sano']"]
    y.write_text("\n".join(lineas) + "\n")
    return y


def verificar_sin_fuga(final: dict[str, list[dict]], umbral_hash: int = 16, tolerancia: float = 0.02,
                       bloquear_visual: bool = True, log=print) -> bool:
    """DURO: ningun original ni grupo en dos splits (siempre bloquea).
    VISUAL: imagenes de val/test/benchmark casi identicas a algo de train. Con fuentes de estudio muy
    uniformes, huevos DISTINTOS en el mismo montaje pueden parecerse: con bloquear_visual=False se
    reporta como advertencia en vez de detener todo."""
    ok = True
    for campo in ("base", "grupo"):
        dueno = {}
        for s, lista in final.items():
            for it in lista:
                previo = dueno.setdefault(it[campo], s)
                if previo != s:
                    ok = False
                    log(f"  FUGA por {campo}: {it[campo]} en {previo} y {s}")
    tr = final.get("train", [])
    if tr:
        Vtr = np.stack([it["hv"] for it in tr])
        for s in ("val", "test", "benchmark"):
            ev = final.get(s, [])
            if not ev:
                continue
            cerca = sum(int(_popcount(it["hv"][0][None, None, :] ^ Vtr).min()) <= umbral_hash for it in ev)
            frac = cerca / len(ev)
            aviso = "" if frac <= tolerancia else ("  <- SUPERA la tolerancia" if bloquear_visual
                                                   else "  <- ADVERTENCIA: escenas parecidas entre splits; el test puede ser algo optimista")
            log(f"  {s:<9} parecidas a algo de train: {cerca}/{len(ev)} = {100 * frac:.1f}% (tolerancia {100 * tolerancia:.0f}%){aviso}")
            if frac > tolerancia and bloquear_visual:
                ok = False
    return ok


# --------------------------------------------------- negativos duros sinteticos
TINTAS = {"roja": (190, 30, 40), "azul": (35, 60, 170), "verde": (30, 120, 60),
          "morada": (110, 40, 140), "negra": (25, 25, 30)}
_FUENTES_TTF = ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                "C:/Windows/Fonts/arialbd.ttf", "C:/Windows/Fonts/arial.ttf"]


def _fuente(tam: int):
    for f in _FUENTES_TTF:
        if Path(f).exists():
            return ImageFont.truetype(f, tam)
    try:
        return ImageFont.load_default(size=tam)
    except TypeError:
        return ImageFont.load_default()


def _texto_sello(rng: random.Random) -> str:
    return rng.choice([
        f"L{rng.randint(1000, 9999)}", f"{rng.randint(1, 28):02d}/{rng.randint(1, 12):02d}",
        f"0CO{rng.randint(1000, 99999)}", f"{rng.randint(0, 3)}ES{rng.randint(10000, 99999)}",
        "A", "AA", f"CAD {rng.randint(1, 28):02d}{rng.choice(['OCT', 'NOV', 'DIC', 'ENE'])}"])


def marcar_huevo(im: Image.Image, caja: tuple, rng: random.Random, estilo: str) -> Image.Image:
    """Dibuja una marca SUPERFICIAL sobre el huevo de `caja` (normalizada). El huevo sigue sano."""
    W, H = im.size
    _, cx, cy, w, h = caja[:5]
    ex, ey, rx, ry = cx * W, cy * H, w * W / 2 * 0.62, h * H / 2 * 0.62      # dentro de la elipse
    if rx < 6 or ry < 6:
        return im
    capa = Image.new("RGBA", im.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(capa)
    alfa = rng.randint(140, 225)
    if estilo == "sello":
        tinta = TINTAS[rng.choice(["roja", "azul", "verde", "morada", "roja", "azul"])]
        fnt = _fuente(max(9, int(ry * rng.uniform(0.28, 0.42))))
        txt = _texto_sello(rng)
        caja_txt = Image.new("RGBA", (int(rx * 2.4), int(ry * 1.2)), (0, 0, 0, 0))
        dt = ImageDraw.Draw(caja_txt)
        dt.text((caja_txt.width / 2, caja_txt.height / 2), txt, fill=tinta + (alfa,), font=fnt, anchor="mm")
        if rng.random() < 0.35:
            dt.ellipse([2, 2, caja_txt.width - 3, caja_txt.height - 3], outline=tinta + (alfa,), width=2)
        caja_txt = caja_txt.rotate(rng.uniform(-30, 30), resample=Image.BICUBIC, expand=True)
        capa.alpha_composite(caja_txt, (int(ex - caja_txt.width / 2), int(ey - caja_txt.height / 2)))
    elif estilo in ("lapicero_color", "lapicero_oscuro"):
        tinta = TINTAS[rng.choice(["azul", "roja", "verde"])] if estilo == "lapicero_color" else TINTAS["negra"]
        for _ in range(rng.choice([1, 1, 2])):
            t0, largo = rng.uniform(0, 2 * math.pi), rng.uniform(0.6, 1.6)
            pts = []
            for k in range(16):
                t = t0 + largo * k / 15
                r = rng.uniform(0.35, 0.9)
                pts.append((ex + rx * r * math.cos(t) + rng.uniform(-2, 2), ey + ry * r * math.sin(t * 1.3) + rng.uniform(-2, 2)))
            d.line(pts, fill=tinta + (alfa,), width=max(2, int(min(rx, ry) * 0.05)), joint="curve")
    else:                                                   # suciedad
        for _ in range(rng.randint(3, 9)):
            px, py = ex + rng.uniform(-rx, rx) * 0.9, ey + rng.uniform(-ry, ry) * 0.9
            s = max(2, int(min(rx, ry) * rng.uniform(0.04, 0.14)))
            tono = (rng.randint(80, 125), rng.randint(60, 90), rng.randint(35, 60), rng.randint(110, 200))
            d.ellipse([px - s, py - s * rng.uniform(0.6, 1.4), px + s, py + s * rng.uniform(0.6, 1.4)], fill=tono)
    capa = capa.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 0.9)))
    out = im.convert("RGBA")
    out.alpha_composite(capa)
    return out.convert("RGB")


def generar_negativos_duros(final_train: list[dict], destino: Path, fraccion: float = 0.35,
                            pesos: dict[str, float] | None = None, semilla: int = 7, log=print) -> Counter:
    """Copias MARCADAS de imagenes de train con huevos sanos. Solo en train. Etiqueta intacta."""
    pesos = pesos or {"sello": 0.40, "lapicero_color": 0.25, "lapicero_oscuro": 0.20, "suciedad": 0.15}
    estilos, w = zip(*pesos.items())
    rng = random.Random(semilla)
    hechos = Counter()
    for it in final_train:
        sanos = [c for c in it["cajas"] if c[0] == "Sano"]
        if not sanos or rng.random() > fraccion:
            continue
        im = Image.open(it["ruta_final"]).convert("RGB")
        usados = []
        for c in rng.sample(sanos, k=min(len(sanos), rng.choice([1, 1, 2]))):
            e = rng.choices(estilos, weights=w)[0]
            im = marcar_huevo(im, c, rng, e)
            usados.append(e)
        nombre = it["ruta_final"].stem + "__neg"
        im.save(destino / "train" / "images" / (nombre + ".jpg"), quality=92)
        escribir_etiqueta(destino / "train" / "labels" / (nombre + ".txt"), it["cajas"])
        hechos.update(usados)
        hechos["imagenes"] += 1
    log(f"  negativos duros: {hechos['imagenes']} imagenes nuevas | marcas: "
        + ", ".join(f"{k}={v}" for k, v in hechos.items() if k != "imagenes"))
    return hechos


# ------------------------------------------------------------ color de cascara
def rasgos_color(im: Image.Image, caja: tuple) -> tuple[float, float]:
    """(saturacion mediana, brillo mediano) del centro del huevo, en [0,1]."""
    W, H = im.size
    _, cx, cy, w, h = caja[:5]
    x1, y1 = int((cx - w * 0.28) * W), int((cy - h * 0.28) * H)
    x2, y2 = int((cx + w * 0.28) * W), int((cy + h * 0.28) * H)
    if x2 - x1 < 3 or y2 - y1 < 3:
        return 0.0, 0.0
    hsv = np.asarray(im.crop((max(0, x1), max(0, y1), min(W, x2), min(H, y2))).convert("HSV"), dtype=np.float32) / 255.0
    return float(np.median(hsv[..., 1])), float(np.median(hsv[..., 2]))


def calibrar_color(muestras: list[tuple[float, float, str]], semilla: int = 0) -> dict:
    """Umbral de saturacion blanco/marron, ajustado en la mitad de las muestras con color
    real y medido en la otra mitad (sin trampa)."""
    rng = random.Random(semilla)
    m = list(muestras)
    rng.shuffle(m)
    mitad = len(m) // 2
    ajuste, prueba = m[:mitad], m[mitad:]

    def exactitud(lista, u):
        if not lista:
            return float("nan")
        ok_b = [s < u for s, _, c in lista if c == "blanco"]
        ok_m = [s >= u for s, _, c in lista if c == "marron"]
        partes = [np.mean(x) for x in (ok_b, ok_m) if x]
        return float(np.mean(partes))            # exactitud balanceada

    candidatos = np.linspace(0.05, 0.6, 111)
    u = float(max(candidatos, key=lambda t: exactitud(ajuste, t))) if ajuste else 0.2
    return {"umbral_saturacion": u, "exactitud_ajuste": exactitud(ajuste, u),
            "exactitud_prueba": exactitud(prueba, u), "n_ajuste": len(ajuste), "n_prueba": len(prueba)}


def ic95(k: int, n: int) -> tuple[float, float]:
    if n == 0:
        return (0.0, 1.0)
    z, p = 1.96, k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, c - m), min(1.0, c + m))

### Módulo `v3_entrenamiento.py` — regla de parada con cierre ordenado y aumentaciones fotométricas
Archivo probado localmente; se escribe tal cual.

In [ ]:
%%writefile v3_entrenamiento.py
# =============================================================================
#  v3 — piezas de entrenamiento (probadas con un entrenamiento REAL en miniatura:
#  v3/prueba_mini_entrenamiento.py, ultralytics 8.4.157)
# =============================================================================
from __future__ import annotations

import csv
import math
import statistics
import time
from pathlib import Path


def aumentaciones_fotometricas() -> list:
    """Transformaciones de Albumentations para las fallas MEDIDAS en campo.

    Por defecto ultralytics aplica desenfoque con p=0.01 y brillo/gamma/JPEG con
    p=0. En campo el modelo 1 perdio el 50 % de los huevos con sobreexposicion y
    mando al empaque rotos desenfocados. Ninguna es geometrica: las cajas no se
    tocan (la geometria del huevo se cuida igual que en el modelo 1).
    """
    import albumentations as A
    return [
        A.MotionBlur(blur_limit=(3, 9), p=0.15),                              # celular movido
        A.Defocus(radius=(1, 4), alias_blur=(0.1, 0.3), p=0.10),              # foto desenfocada
        A.RandomBrightnessContrast(brightness_limit=(-0.25, 0.35), contrast_limit=0.25, p=0.30),
        A.RandomGamma(gamma_limit=(70, 150), p=0.15),                         # sobre / sub exposicion
        A.ImageCompression(quality_range=(40, 95), p=0.20),                   # JPEG del celular
        A.GaussNoise(std_range=(0.02, 0.06), p=0.10),                         # poca luz = ruido
        A.CLAHE(p=0.05),
        A.ToGray(p=0.02),
    ]


def _lf_original(args) -> callable:
    """Replica el scheduler de ultralytics 8.4.157 (trainer._setup_scheduler)."""
    n, lrf = int(args.epochs), float(args.lrf)
    if args.cos_lr:
        return lambda x: max((1 - math.cos(x * math.pi / n)) / 2, 0) * (lrf - 1) + 1
    return lambda x: max(1 - x / n, 0) * (1.0 - lrf) + lrf


def crear_parada_ordenada(min_epocas: int, paciencia: int, mejora_minima: float, epocas_finales: int,
                          columna: str = "mAP50-95", log=print):
    """Regla de las sesiones anteriores, con un cierre ORDENADO en vez de un corte seco.

    - Nunca decide antes de `min_epocas`.
    - Despues, si pasan `paciencia` epocas sin una mejora de al menos `mejora_minima`
      en `columna` (mAP50-95: es el fitness de deteccion en 8.4.157), no corta: entra
      en una FASE FINAL de `epocas_finales` epocas sin mosaico y con la tasa de
      aprendizaje bajando a su minimo, y luego termina. Cortar en seco dejaria el LR
      alto y el mosaico abierto, que es justo donde se ganan los ultimos puntos.
    - Si nunca hay meseta, el entrenamiento llega al tope con el cierre normal.
    - Todo se recalcula desde results.csv: reanudar tras una desconexion da la misma
      decision. Engancharla con `registrar` (tambien al inicio): asi una reanudacion
      en plena fase final arranca ya sin mosaico y con el LR de la fase final.
    - El tope (`epochs` de train) se respeta: si la meseta llega en las ultimas
      `epocas_finales`, el cierre normal de ultralytics ya esta en marcha.
    - Cada linea trae el tiempo real por epoca y cuanto falta: ultralytics solo
      muestra el de la epoca en curso, y en Colab hay que planear las sesiones.

    Requiere entrenar con close_mosaic=epocas_finales y patience=0.
    """
    reloj = {"t": None, "dur": []}

    def restante(desde: int, hasta: int) -> str:
        if not reloj["dur"]:
            return ""
        s = statistics.median(reloj["dur"][-10:])
        return f" (~{max(hasta - desde, 0) * s / 3600:.1f} h)"

    def parada(trainer):
        ahora = time.time()
        if reloj["t"] is not None:
            reloj["dur"].append(ahora - reloj["t"])
        reloj["t"] = ahora
        ruta = Path(trainer.csv)
        if not ruta.exists():
            return
        with open(ruta, encoding="utf-8") as fh:
            filas = [{k.strip(): v for k, v in f.items()} for f in csv.DictReader(fh)]
        col = next((k for k in filas[0] if columna in k), None) if filas else None
        if col is None:
            return
        hist = [float(f[col]) for f in filas]
        ref, ref_ep, t = hist[0], 1, None
        for ep, v in enumerate(hist, 1):
            if v > ref + mejora_minima:
                ref, ref_ep = v, ep
            if t is None and ep >= min_epocas and ep - ref_ep >= paciencia:
                t = ep                                   # primera epoca en que se cumple la regla
        epoca, tope = len(hist), int(trainer.args.epochs)
        ritmo = f" | {statistics.median(reloj['dur'][-10:]) / 60:.1f} min/epoca" if reloj["dur"] else ""
        if t is None:
            extra = f" | minimo {min_epocas}" if epoca < min_epocas else ""
            fin_min = min(max(min_epocas, ref_ep + paciencia) + epocas_finales, tope)
            eta = (f": termina entre la {fin_min}{restante(epoca, fin_min)} y la {tope}{restante(epoca, tope)}"
                   if ritmo else "")
            log(f"  [parada] epoca {epoca}: {columna}={hist[-1]:.4f} | referencia {ref:.4f} (epoca {ref_ep})"
                f" | {epoca - ref_ep}/{paciencia} sin mejorar >= {mejora_minima:g}{extra}{ritmo}{eta}")
            return
        fin = min(t + epocas_finales, tope)
        if fin == tope and trainer.epochs == tope:
            log(f"  [parada] epoca {epoca}: meseta en la recta final; el cierre normal termina en la {tope}"
                f"{restante(epoca, tope)}.")
        elif trainer.epochs != fin:
            base = _lf_original(trainer.args)
            lrf, f0 = float(trainer.args.lrf), base(t - 1)

            def lf(x, t=t, f0=f0, base=base):
                if x < t:
                    return base(x)
                p = min(1.0, (x - t + 1) / epocas_finales)
                return lrf + (f0 - lrf) * 0.5 * (1 + math.cos(math.pi * p))

            trainer.lf = lf
            trainer.scheduler.lr_lambdas = [lf] * len(trainer.scheduler.lr_lambdas)
            trainer.epochs = fin
            log(f"  [parada] meseta desde la epoca {t} (ultima mejora significativa: {ref_ep}). "
                f"FASE FINAL de {epocas_finales} epocas: sin mosaico, LR -> minimo. "
                f"Termina en la {fin}{restante(epoca, fin)}.")
        else:
            log(f"  [parada] fase final: epoca {epoca} de {fin}{ritmo}{restante(epoca, fin)}")
        if epoca >= fin:
            trainer.stop = True

    return parada


def registrar(modelo, parada) -> None:
    """Engancha la regla al inicio del entrenamiento y al final de cada epoca.

    Al inicio importa al REANUDAR: ultralytics decide el mosaico y el LR al empezar
    cada epoca, asi que sin esto la primera epoca reanudada dentro de la fase final
    correria con mosaico y con el LR original.
    """
    modelo.add_callback("on_train_start", parada)
    modelo.add_callback("on_fit_epoch_end", parada)

### Módulo `v3_inferencia.py` — inferencia idéntica a la del backend (probada: probabilidades iguales bit a bit)
Archivo probado localmente; se escribe tal cual.

In [ ]:
%%writefile v3_inferencia.py
# =============================================================================
#  v3 — inferencia IDENTICA a la del backend (backend/app/detector.py + routing.py)
#
#  "Evaluar lo que se despliega": el notebook mide con el mismo letterbox, el
#  mismo NMS en numpy, las mismas dos probabilidades por huevo y la misma
#  compuerta de nitidez que corren en el servidor. Probado contra el backend
#  real con v3/validar_notebook_v3.py (mismas cajas, mismas probabilidades).
# =============================================================================
from __future__ import annotations

import numpy as np
from PIL import Image


def letterbox(im: Image.Image, imgsz: int = 640):
    w, h = im.size
    r = min(imgsz / w, imgsz / h)
    nw, nh = int(round(w * r)), int(round(h * r))
    lienzo = Image.new("RGB", (imgsz, imgsz), (114, 114, 114))
    dw, dh = (imgsz - nw) // 2, (imgsz - nh) // 2
    lienzo.paste(im.resize((nw, nh), Image.BILINEAR), (dw, dh))
    x = np.asarray(lienzo, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    return np.ascontiguousarray(x), r, dw, dh


def _nms(cajas, scores, iou_thr):
    x1, y1, x2, y2 = cajas.T
    areas = (x2 - x1) * (y2 - y1)
    orden = scores.argsort()[::-1]
    keep = []
    while orden.size:
        i = orden[0]
        keep.append(int(i))
        if orden.size == 1:
            break
        xx1, yy1 = np.maximum(x1[i], x1[orden[1:]]), np.maximum(y1[i], y1[orden[1:]])
        xx2, yy2 = np.minimum(x2[i], x2[orden[1:]]), np.minimum(y2[i], y2[orden[1:]])
        inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
        orden = orden[1:][inter / (areas[i] + areas[orden[1:]] - inter + 1e-9) <= iou_thr]
    return np.asarray(keep, dtype=int)


def postprocesar(salida: np.ndarray, r, dw, dh, W, H, conf_min: float, iou_nms: float) -> list[tuple]:
    """-> [(x1, y1, x2, y2, p_roto, p_sano)] en pixeles del original. Igual que el backend."""
    p = salida[0]
    if p.shape[0] < p.shape[1]:
        p = p.T
    xywh, scores = p[:, :4], p[:, 4:]
    mejor = scores.max(1)
    m = mejor >= conf_min
    if not m.any():
        return []
    xywh, scores, mejor = xywh[m], scores[m], mejor[m]
    cx, cy, w, h = xywh.T
    b = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)
    b[:, [0, 2]] = ((b[:, [0, 2]] - dw) / r).clip(0, W)
    b[:, [1, 3]] = ((b[:, [1, 3]] - dh) / r).clip(0, H)
    idx = scores.argmax(1)
    keep = []
    for c in np.unique(idx):
        k = np.flatnonzero(idx == c)
        keep.extend(k[_nms(b[k], mejor[k], iou_nms)].tolist())
    keep.sort(key=lambda i: -mejor[i])
    return [(float(b[i, 0]), float(b[i, 1]), float(b[i, 2]), float(b[i, 3]), float(scores[i, 0]), float(scores[i, 1]))
            for i in keep]


def nitidez_recorte(im: Image.Image, x1, y1, x2, y2, lado: int = 192) -> float:
    a, b = max(0, int(x1)), max(0, int(y1))
    c, d = min(im.width, int(x2)), min(im.height, int(y2))
    if c - a < 8 or d - b < 8:
        return 0.0
    g = np.asarray(im.crop((a, b, c, d)).convert("L").resize((lado, lado), Image.BILINEAR), dtype=np.float32)
    lap = 4 * g[1:-1, 1:-1] - g[:-2, 1:-1] - g[2:, 1:-1] - g[1:-1, :-2] - g[1:-1, 2:]
    return float(lap.var())


def decidir(p_roto, p_sano, nitidez, tau_roto, tau_sano, nitidez_min=4.0) -> str:
    """Politica del backend (routing.MotorDecision): DESCARTE / ACEPTADO / REVISION."""
    if nitidez is not None and nitidez < nitidez_min and p_roto < tau_roto:
        return "REVISION"
    if p_roto >= tau_roto:
        return "DESCARTE"
    if p_sano >= tau_sano:
        return "ACEPTADO"
    return "REVISION"


def iou(a, b) -> float:
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    i = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-9)


def emparejar(gts: list, preds: list, umbral: float = 0.5) -> list:
    """Cada huevo real -> la deteccion con mayor IoU (>= umbral), sin repetir. None si no se detecto."""
    usadas, out = set(), []
    for g in gts:
        mejor, mi = None, umbral
        for k, p in enumerate(preds):
            v = iou(g[1:5], p[:4])
            if v >= mi and k not in usadas:
                mejor, mi = k, v
        if mejor is not None:
            usadas.add(mejor)
        out.append(None if mejor is None else preds[mejor])
    return out

In [ ]:
# @title 3. Descargar las fuentes
import zipfile, shutil, importlib
import v3_datos_lib as L; importlib.reload(L)
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))

if FUENTES_DIR.exists(): shutil.rmtree(FUENTES_DIR)
FUENTES_DIR.mkdir(parents=True)
disponibles = []
for f in FUENTES:
    destino = FUENTES_DIR / f["nombre"]
    try:
        if "local" in f:
            with zipfile.ZipFile(f["local"]) as z:
                z.extractall(destino)
        else:
            proy = rf.workspace(f["ws"]).project(f["proyecto"])
            proy.version(f["version"]).download("yolov8", location=str(destino), overwrite=True)
            print(f"  licencia de {f['nombre']}: {getattr(proy, 'license', '?')}")
        disponibles.append(f)
        print(f"OK  {f['nombre']}")
    except Exception as e:
        print(f"--  {f['nombre']}: no disponible ({type(e).__name__}: {str(e)[:120]}). Se sigue sin ella.")
assert any(f["nombre"] == "huevos" for f in disponibles), "Falta huevos.zip en MyDrive/eggdetection/: es el benchmark comun"

In [ ]:
# @title 4. Auditoría por fuente: convención de etiquetado y qué se excluye
import pandas as pd
from collections import Counter
items, filas, auditoria = [], [], {}
for f in disponibles:
    its = L.inventariar(FUENTES_DIR / f["nombre"], f["nombre"])
    au = L.auditar(its, f["mapa"])
    conv = au["convencion"] if f["convencion"] == "auto" else f["convencion"]
    st = L.mapear(its, f["mapa"], conv, f.get("color"))
    buenos = [it for it in its if not it["excluir"] and it["cajas"]]
    auditoria[f["nombre"]] = {**au, "convencion_usada": conv, "descartes": st}
    filas.append({"fuente": f["nombre"], "imagenes": len(its), "originales": au["originales"],
                  "convencion": conv + (" (auto)" if f["convencion"] == "auto" else ""),
                  "IoA roto/sano": au["ioa_roto_en_sano"], "excluidas": st.get("imagenes_excluidas", 0),
                  "usables": len(buenos), "roto": sum(c[0] == "Roto" for it in buenos for c in it["cajas"]),
                  "sano": sum(c[0] == "Sano" for it in buenos for c in it["cajas"])})
    sin = {k: v for k, v in st.items() if k.startswith("clase sin mapeo")}
    if sin:
        print(f"ATENCION {f['nombre']}: clases no previstas -> imagenes excluidas: {sin}")
    items += buenos
display(pd.DataFrame(filas).set_index("fuente"))
print("\n'region' = la fuente encierra solo la GRIETA dentro de una caja de huevo: se convierte a caja de huevo roto.")
print("Clases por fuente (cajas, area mediana relativa):")
for n, a in auditoria.items():
    print(f"  {n:<8} " + " | ".join(f"{k}: {v['cajas']} ({v['area_mediana']:.3f})" for k, v in a["clases"].items()))

In [ ]:
# @title 5. Deduplicar entre fuentes, agrupar y repartir (con benchmark intocable)
import csv
L.calcular_hashes(items)
items = L.deduplicar_entre_fuentes(items, PRIORIDAD, umbral=4)
info = L.agrupar(items, umbral_hash=16, max_grupo_hash=30)
print("uniones 'mismo huevo':", dict(info["razones"]), "| rechazadas por tope:", info["rechazadas_por_tope"])

es_benchmark = lambda it: it["fuente"] == "huevos" and it["split_orig"] == "test"
n_orig = len({it["base"] for it in items if not es_benchmark(it)})
PROPORCIONES = {"train": 0.80, "val": 0.10, "test": 0.10} if n_orig >= 3000 else {"train": 0.70, "val": 0.15, "test": 0.15}
print(f"\n{n_orig} originales independientes (sin benchmark) -> split "
      + "/".join(f"{int(100 * v)}" for v in PROPORCIONES.values()))

asignado = L.repartir(items, PROPORCIONES, benchmark=es_benchmark, semilla=SEMILLA)
reparto = L.revisar_reparto(items, asignado, PROPORCIONES)     # se detiene si queda desbalanceado
final = L.materializar(items, asignado, DATASET)
data_yaml = L.escribir_data_yaml(DATASET, list(final))
print("reparto de originales:", {k: f"{100 * v:.1f}%" for k, v in reparto.items()})
for s in ("train", "val", "test", "benchmark"):
    lst = final.get(s, [])
    print(f"  {s:<9} {len(lst):>5} imagenes | Roto {sum(c[0]=='Roto' for it in lst for c in it['cajas']):>5}"
          f" | Sano {sum(c[0]=='Sano' for it in lst for c in it['cajas']):>5} | fuentes {dict(Counter(it['fuente'] for it in lst))}")
print("\n=== VERIFICACION ===")
assert L.verificar_sin_fuga(final, bloquear_visual=False), "Fuga DURA: un original o grupo quedo en dos splits"
print("SIN FUGA DURA entre train, val, test y benchmark (el chequeo visual de arriba es informativo)")
with open(TRABAJO / "manifiesto_v3.csv", "w", newline="") as fh:
    w = csv.writer(fh); w.writerow(["split", "fuente", "archivo", "original", "grupo", "roto", "sano"])
    for s, lst in final.items():
        for it in lst:
            w.writerow([s, it["fuente"], it["ruta_final"].name, it["base"], it["grupo"],
                        sum(c[0] == "Roto" for c in it["cajas"]), sum(c[0] == "Sano" for c in it["cajas"])])

In [ ]:
# @title 6. Color de cáscara: calibrado con las fuentes que traen el color real
from PIL import Image
muestras = []
for s in ("train", "val"):
    for it in final.get(s, []):
        cc = [c for c in it["cajas"] if c[5]]
        if cc:
            im = Image.open(it["ruta_final"]).convert("RGB")
            muestras += [(*L.rasgos_color(im, c), c[5]) for c in cc]
if len(muestras) >= 40:
    COLOR = L.calibrar_color(muestras)
    print(f"umbral de saturacion {COLOR['umbral_saturacion']:.2f} | exactitud en la mitad NO usada para ajustar: "
          f"{100 * COLOR['exactitud_prueba']:.1f}% (n={COLOR['n_prueba']})")
    if COLOR["exactitud_prueba"] < 0.9:
        print("ATENCION: exactitud < 90%: el desglose por color es orientativo.")
else:
    COLOR = None
    print("Sin fuentes con color real disponibles: se omite el desglose por color.")

def color_de(im, caja):
    if COLOR is None:
        return "?"
    return "marron" if L.rasgos_color(im, caja)[0] >= COLOR["umbral_saturacion"] else "blanco"

In [ ]:
# @title 7. Negativos duros sintéticos (solo train) y vista previa
import matplotlib.pyplot as plt, random
NEG = L.generar_negativos_duros(final["train"], DATASET, fraccion=FRACCION_NEGATIVOS, semilla=SEMILLA)
negs = sorted((DATASET / "train" / "images").glob("*__neg.jpg"))
random.seed(SEMILLA)
fig, ejes = plt.subplots(2, 6, figsize=(18, 6.5))
for ax, f in zip(ejes.flat, random.sample(negs, min(12, len(negs)))):
    ax.imshow(Image.open(f)); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Negativos duros: huevos SANOS con sello, raya o suciedad (etiquetados como Sano)"); plt.tight_layout(); plt.show()

In [ ]:
# @title 8. Vista previa con cajas, una fila por fuente (revisa que las etiquetas tengan sentido)
from PIL import ImageDraw
fuentes_vistas = sorted({it["fuente"] for it in final["train"]})
fig, ejes = plt.subplots(len(fuentes_vistas), 5, figsize=(17, 3.4 * len(fuentes_vistas)), squeeze=False)
for fila, fu in enumerate(fuentes_vistas):
    cand = [it for it in final["train"] if it["fuente"] == fu]
    for col, it in enumerate(random.sample(cand, min(5, len(cand)))):
        im = Image.open(it["ruta_final"]).convert("RGB"); d = ImageDraw.Draw(im); W, H = im.size
        for c in it["cajas"]:
            x1, y1, x2, y2 = (c[1] - c[3] / 2) * W, (c[2] - c[4] / 2) * H, (c[1] + c[3] / 2) * W, (c[2] + c[4] / 2) * H
            d.rectangle([x1, y1, x2, y2], outline="#d03b3b" if c[0] == "Roto" else "#0ca30c", width=max(2, W // 160))
        ax = ejes[fila][col]; ax.imshow(im); ax.set_xticks([]); ax.set_yticks([])
        if col == 0: ax.set_ylabel(fu, fontsize=12)
plt.suptitle("Rojo = Roto, verde = Sano"); plt.tight_layout(); plt.show()

In [ ]:
# @title 9. Desbalance de clases -> peso por clase
n_roto = sum(c[0] == "Roto" for it in final["train"] for c in it["cajas"])
n_sano = sum(c[0] == "Sano" for it in final["train"] for c in it["cajas"]) + NEG["imagenes"]
razon = max(n_roto, n_sano) / max(1, min(n_roto, n_sano))
CLS_PW = 0.5 if razon > 1.5 else 0.0   # cls_pw SI funciona en deteccion (8.4.157), a diferencia de clasificacion
print(f"train: Roto {n_roto} | Sano ~{n_sano} (con negativos) | razon {razon:.2f} -> cls_pw={CLS_PW}")

In [ ]:
# @title 10. Módulos de entrenamiento e inferencia
import v3_entrenamiento as V, v3_inferencia as VI
importlib.reload(V); importlib.reload(VI)
print("aumentaciones:", ", ".join(type(t).__name__ for t in V.aumentaciones_fotometricas()))

In [ ]:
# @title 11. Entrenar (o reanudar): mínimo 150 épocas, cierre ordenado tras la meseta
# Si Colab se desconecto: corre 1-10 (recrean el mismo dataset: todo usa SEMILLA), pega aqui el nombre
# de la carpeta del run en MyDrive/eggdetection/runs_v3/ y sigue desde esta celda.
REANUDAR = ""  # @param {type:"string"}
from ultralytics import YOLO
parada = V.crear_parada_ordenada(MIN_EPOCAS, PACIENCIA, MEJORA_MINIMA, EPOCAS_FINALES)
if REANUDAR:
    RUN = CARPETA_RUNS / REANUDAR
    modelo = YOLO(str(RUN / "weights" / "last.pt"))
    V.registrar(modelo, parada)   # al inicio (reanudar en plena fase final) y al final de cada epoca
    modelo.train(resume=True)
else:
    modelo = YOLO(MODELO_BASE)
    V.registrar(modelo, parada)   # al inicio (reanudar en plena fase final) y al final de cada epoca
    modelo.train(
        data=str(data_yaml), imgsz=IMGSZ, batch=BATCH, epochs=EPOCAS_MAX,
        patience=0, close_mosaic=EPOCAS_FINALES,             # la regla y el cierre los maneja el callback
        optimizer="AdamW", lr0=0.001, lrf=0.01, cos_lr=True, warmup_epochs=3, weight_decay=0.0005,
        box=7.5, cls=0.7, dfl=1.5, cls_pw=CLS_PW,
        # geometria del huevo intacta: sin shear, perspectiva ni mixup (igual que el modelo 1)
        degrees=10.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0, flipud=0.5, fliplr=0.5,
        mosaic=1.0, mixup=0.0, copy_paste=0.0, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
        augmentations=V.aumentaciones_fotometricas(),
        cache="disk", workers=2, seed=SEMILLA, deterministic=False, amp=True,
        project=str(CARPETA_RUNS), name=NOMBRE_RUN, exist_ok=True, plots=True,
    )
    RUN = Path(modelo.trainer.save_dir)
print("\npesos:", RUN / "weights" / "best.pt")

In [ ]:
# @title 12. Curvas de entrenamiento
df = pd.read_csv(RUN / "results.csv"); df.columns = [c.strip() for c in df.columns]
c50 = next(c for c in df.columns if "mAP50(B)" in c); c95 = next(c for c in df.columns if "mAP50-95" in c)
fig, (a, b) = plt.subplots(1, 2, figsize=(14, 4))
a.plot(df["epoch"], df[c50], label="mAP50"); a.plot(df["epoch"], df[c95], label="mAP50-95"); a.legend(); a.set_title("val")
for k in ("train/box_loss", "train/cls_loss", "val/box_loss", "val/cls_loss"):
    if k in df: b.plot(df["epoch"], df[k], label=k)
b.legend(); b.set_title("pérdidas"); plt.tight_layout(); plt.show()
i = int(df[c95].idxmax())
print(f"{len(df)} epocas | mejor mAP50-95 {df.loc[i, c95]:.4f} (mAP50 {df.loc[i, c50]:.4f}) en la epoca {int(df.loc[i, 'epoch'])}")

In [ ]:
# @title 13. Evaluación como en producción: umbrales calibrados en VAL, medidos en TEST y BENCHMARK
import numpy as np, torch
DISP = "cuda" if torch.cuda.is_available() else "cpu"
modelo = YOLO(str(RUN / "weights" / "best.pt"))
red = modelo.model.float().eval().to(DISP)

def predecir(im):
    x, r, dw, dh = VI.letterbox(im, IMGSZ)
    with torch.no_grad():
        o = red(torch.from_numpy(x).to(DISP)); o = o[0] if isinstance(o, (list, tuple)) else o
    return VI.postprocesar(o.float().cpu().numpy(), r, dw, dh, im.width, im.height, CONF_MIN, IOU_NMS)

def huevos_de(it, im):
    W, H = im.size
    return [(c[0], (c[1] - c[3] / 2) * W, (c[2] - c[4] / 2) * H, (c[1] + c[3] / 2) * W, (c[2] + c[4] / 2) * H, c)
            for c in it["cajas"]]

def recolectar(split, transform=None, solo=None):
    filas = []
    for it in final.get(split, []):
        im = Image.open(it["ruta_final"]).convert("RGB")
        gts = huevos_de(it, im)
        if solo and not any(g[0] == solo for g in gts): continue
        im_t = transform(im, gts) if transform else im
        preds = predecir(im_t)
        for g, p in zip(gts, VI.emparejar(gts, preds)):
            if solo and g[0] != solo: continue
            nit = VI.nitidez_recorte(im_t, *p[:4]) if p else None
            filas.append({"clase": g[0], "fuente": it["fuente"], "color": color_de(im, g[5]),
                          "p_roto": p[4] if p else None, "p_sano": p[5] if p else None, "nitidez": nit})
    return filas

VAL = recolectar("val")
roto = [f for f in VAL if f["clase"] == "Roto"]; sano = [f for f in VAL if f["clase"] == "Sano"]
mejor = None
for tau in np.round(np.arange(0.05, 0.96, 0.01), 2):
    fn = sum(f["p_roto"] is None or f["p_roto"] < tau for f in roto)
    fp = sum(f["p_roto"] is not None and f["p_roto"] >= tau for f in sano)
    c = fn * COSTO_FN + fp * COSTO_FP
    if mejor is None or c < mejor[0]: mejor = (c, float(tau))
TAU_ROTO = mejor[1]
ps = sorted(f["p_sano"] for f in sano if f["p_sano"] is not None and f["p_roto"] < TAU_ROTO)
TAU_SANO = float(min(0.95, max(0.50, np.percentile(ps, 100 * MAX_REVISION_SANOS)))) if ps else 0.85
print(f"calibrado en VAL: TAU_ROTO={TAU_ROTO:.2f} (costo minimo {COSTO_FN:g}:{COSTO_FP:g}) | TAU_SANO={TAU_SANO:.2f}")

def rutear(f):
    if f["p_roto"] is None: return "NO_DETECTADO"
    return VI.decidir(f["p_roto"], f["p_sano"], f["nitidez"], TAU_ROTO, TAU_SANO, NITIDEZ_MIN)

def tabla(filas, titulo):
    print(f"\n{titulo}")
    print(f"  {'real':<6}{'DESCARTE':>10}{'REVISION':>10}{'ACEPTADO':>10}{'no det.':>9}{'n':>6}")
    res = {}
    for cl in ("Roto", "Sano"):
        fs = [f for f in filas if f["clase"] == cl]; r = Counter(rutear(f) for f in fs)
        res[cl] = {k: r[k] for k in ("DESCARTE", "REVISION", "ACEPTADO", "NO_DETECTADO")} | {"n": len(fs)}
        print(f"  {cl:<6}{r['DESCARTE']:>10}{r['REVISION']:>10}{r['ACEPTADO']:>10}{r['NO_DETECTADO']:>9}{len(fs):>6}")
    ro = res["Roto"]; lo, hi = L.ic95(ro["ACEPTADO"], ro["n"])
    print(f"  rotos al EMPAQUE: {ro['ACEPTADO']}/{ro['n']} = {100*ro['ACEPTADO']/max(1,ro['n']):.2f}% IC95 [{100*lo:.1f}, {100*hi:.1f}]"
          f" | sanos botados: {res['Sano']['DESCARTE']}/{res['Sano']['n']}")
    return res

METRICAS = {"tau_roto": TAU_ROTO, "tau_sano": TAU_SANO, "conf_min": CONF_MIN, "nitidez_min": NITIDEZ_MIN}
for s in ("test", "benchmark"):
    filas = recolectar(s)
    METRICAS[s] = tabla(filas, f"=== {s.upper()} ({len(final.get(s, []))} imagenes) ===")
    if s == "test":
        print("\n  por fuente (rotos al empaque / rotos | sanos botados / sanos):")
        for fu in sorted({f["fuente"] for f in filas}):
            ro = [f for f in filas if f["fuente"] == fu and f["clase"] == "Roto"]; sa = [f for f in filas if f["fuente"] == fu and f["clase"] == "Sano"]
            print(f"    {fu:<8} {sum(rutear(f)=='ACEPTADO' for f in ro):>3}/{len(ro):<4} | {sum(rutear(f)=='DESCARTE' for f in sa):>3}/{len(sa)}")
        if COLOR:
            print("  por color de cascara:")
            for col in ("blanco", "marron"):
                ro = [f for f in filas if f["color"] == col and f["clase"] == "Roto"]; sa = [f for f in filas if f["color"] == col and f["clase"] == "Sano"]
                print(f"    {col:<7} rotos al empaque {sum(rutear(f)=='ACEPTADO' for f in ro)}/{len(ro)} | sanos botados {sum(rutear(f)=='DESCARTE' for f in sa)}/{len(sa)}")
for s in ("test", "benchmark"):
    yml = DATASET / f"data_{s}.yaml"
    yml.write_text(f"path: {DATASET.as_posix()}\ntrain: train/images\nval: {s}/images\nnc: 2\nnames: ['Roto', 'Sano']\n")
    m = modelo.val(data=str(yml), split="val", imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False)
    METRICAS[s + "_map"] = {"mAP50": float(m.box.map50), "mAP50-95": float(m.box.map)}
    print(f"{s}: mAP50={m.box.map50:.4f}  mAP50-95={m.box.map:.4f}")

In [ ]:
# @title 14. Robustez: las pruebas que tumbaron al modelo 1
from PIL import ImageFilter, ImageEnhance, ImageFont

def rayar(im, gts):   # EXACTAMENTE la prueba de diagnostico_campo.py (no es el generador de train)
    im = im.copy(); d = ImageDraw.Draw(im); rng = random.Random(3)
    for g in gts:
        if g[0] != "Sano": continue
        x1, y1, x2, y2 = g[1:5]; cx, alto = (x1 + x2) / 2, y2 - y1
        pts = [(cx + (x2 - x1) * 0.12 * np.sin(t * 3.1) + rng.uniform(-3, 3), y1 + alto * (0.12 + 0.76 * t)) for t in (i / 13 for i in range(14))]
        d.line(pts, fill=(35, 25, 20), width=3, joint="curve")
    return im

def sellar(im, gts):  # sello con fuente, texto y tintas DISTINTOS a los de train
    im = im.copy(); d = ImageDraw.Draw(im)
    for g in gts:
        if g[0] != "Sano": continue
        x1, y1, x2, y2 = g[1:5]
        fnt = ImageFont.load_default(size=max(10, int((y2 - y1) * 0.13)))
        d.text(((x1 + x2) / 2, (y1 + y2) / 2), "EXP 12-26 F7", fill=(200, 110, 20), font=fnt, anchor="mm")
    return im

PRUEBAS = [("raya de lapicero en SANOS", rayar, "Sano"), ("sello de tinta en SANOS", sellar, "Sano"),
           ("desenfoque r=3 en ROTOS", lambda im, g: im.filter(ImageFilter.GaussianBlur(3)), "Roto"),
           ("sobreexposicion 2.4x en ROTOS", lambda im, g: ImageEnhance.Brightness(im).enhance(2.4), "Roto"),
           ("subexposicion 0.35x en ROTOS", lambda im, g: ImageEnhance.Brightness(im).enhance(0.35), "Roto")]
ROBUSTEZ = {}
for s in ("test", "benchmark"):
    print(f"\n=== {s.upper()} ===")
    print(f"  {'prueba':<32}{'DESCARTE':>10}{'REVISION':>10}{'ACEPTADO':>10}{'no det.':>9}{'n':>6}")
    for nombre, tf, cl in PRUEBAS:
        r = Counter(rutear(f) for f in recolectar(s, tf, cl)); n = sum(r.values())
        ROBUSTEZ[f"{s}|{nombre}"] = dict(r) | {"n": n}
        print(f"  {nombre:<32}{r['DESCARTE']:>10}{r['REVISION']:>10}{r['ACEPTADO']:>10}{r['NO_DETECTADO']:>9}{n:>6}")
print("\nModelo 1 en el BENCHMARK (docs/COMPARACION_MODELOS.md):")
print("  raya en sanos -> DESCARTE 24/43 | desenfoque r3 -> 52 desc / 63 rev / 3 empaque / 22 no det (de 140)")
print("  sobreexposicion 2.4x -> 45 desc / 17 rev / 8 empaque / 70 no det (de 140)")

In [ ]:
# @title 15. Exportar ONNX para el backend y verificar que da lo mismo
import onnxruntime as ort
onnx_path = modelo.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False, nms=False)
s = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
forma = s.get_outputs()[0].shape; nombres = s.get_modelmeta().custom_metadata_map.get("names")
print("salida:", forma, "| clases:", nombres)
assert list(forma) == [1, 6, 8400] and nombres == "{0: 'Roto', 1: 'Sano'}", "No es compatible con el backend"
dif = 0.0
for it in final["benchmark"][:25]:
    im = Image.open(it["ruta_final"]).convert("RGB"); x, r, dw, dh = VI.letterbox(im, IMGSZ)
    a = VI.postprocesar(s.run(None, {s.get_inputs()[0].name: x})[0], r, dw, dh, im.width, im.height, CONF_MIN, IOU_NMS)
    b = predecir(im)
    dif = max([dif] + [abs(p[4] - q[4]) for p, q in zip(a, b)] + [abs(p[5] - q[5]) for p, q in zip(a, b)])
print(f"ONNX (CPU, como el servidor) vs PyTorch: dif. maxima de probabilidad = {dif:.2e}")
assert dif < 5e-3, "El ONNX no reproduce al modelo"

In [ ]:
# @title 16. Guardar el paquete en Drive
import json
PAQ = TRABAJO / RUN.name
if PAQ.exists(): shutil.rmtree(PAQ)
(PAQ / "weights").mkdir(parents=True)
shutil.copy2(RUN / "weights" / "best.pt", PAQ / "weights" / "best.pt"); shutil.copy2(onnx_path, PAQ / "weights" / "best.onnx")
for f in ("results.csv", "results.png", "args.yaml", "confusion_matrix.png", "confusion_matrix_normalized.png", "BoxPR_curve.png"):
    if (RUN / f).exists(): shutil.copy2(RUN / f, PAQ / f)
shutil.copy2(TRABAJO / "manifiesto_v3.csv", PAQ / "manifiesto_v3.csv"); shutil.copy2(data_yaml, PAQ / "data.yaml")
(PAQ / "metricas_v3.json").write_text(json.dumps({**METRICAS, "robustez": ROBUSTEZ, "color": COLOR,
    "negativos_duros": dict(NEG), "auditoria": {k: {kk: vv for kk, vv in v.items() if kk != "clases"} for k, v in auditoria.items()},
    "backend_env": {"TAU_ROTO": TAU_ROTO, "TAU_SANO": TAU_SANO, "CONF_MIN_DETECCION": CONF_MIN}}, indent=2, default=str))
zip_path = shutil.make_archive(str(PAQ), "zip", PAQ)
dest = Path("/content/drive/MyDrive/eggdetection"); shutil.copy2(zip_path, dest)
print("guardado:", dest / Path(zip_path).name, f"({Path(zip_path).stat().st_size / 1e6:.0f} MB)")

In [ ]:
# @title 17. Resumen: v3 contra el modelo 1 en el BENCHMARK común
b = METRICAS["benchmark"]; r, s_ = b["Roto"], b["Sano"]
print("=" * 72); print(f"MODELO {RUN.name}"); print("=" * 72)
print(f"{'en el benchmark (mismos 183 huevos)':<40}{'modelo 1':>12}{'v3':>12}")
print(f"{'rotos al empaque':<40}{'0/140':>12}{str(r['ACEPTADO']) + '/' + str(r['n']):>12}")
print(f"{'rotos no detectados':<40}{'8/140':>12}{str(r['NO_DETECTADO']) + '/' + str(r['n']):>12}")
print(f"{'sanos botados':<40}{'0/43':>12}{str(s_['DESCARTE']) + '/' + str(s_['n']):>12}")
rb = ROBUSTEZ.get("benchmark|raya de lapicero en SANOS", {})
print(f"{'sanos con raya -> DESCARTE':<40}{'24/43':>12}{str(rb.get('DESCARTE', 0)) + '/' + str(rb.get('n', 0)):>12}")
bl = ROBUSTEZ.get("benchmark|desenfoque r=3 en ROTOS", {})
print(f"{'rotos desenfocados al empaque':<40}{'3/140':>12}{str(bl.get('ACEPTADO', 0)) + '/' + str(bl.get('n', 0)):>12}")
ov = ROBUSTEZ.get("benchmark|sobreexposicion 2.4x en ROTOS", {})
print(f"{'rotos sobreexpuestos no detectados':<40}{'70/140':>12}{str(ov.get('NO_DETECTADO', 0)) + '/' + str(ov.get('n', 0)):>12}")
print("=" * 72)
print(f"Para el backend (.env): TAU_ROTO={TAU_ROTO:.2f}  TAU_SANO={TAU_SANO:.2f}  CONF_MIN_DETECCION={CONF_MIN}")

In [ ]:
# @title 18. Prueba con TUS fotos
import io
from PIL import ImageOps
from google.colab import files
COL = {"DESCARTE": "#d03b3b", "REVISION": "#fab219", "ACEPTADO": "#0ca30c"}
ETQ = {"DESCARTE": "✗ DAÑADO", "REVISION": "⚠ REVISIÓN", "ACEPTADO": "✓ BUENO"}
subidas = files.upload()
imgs = [ImageOps.exif_transpose(Image.open(io.BytesIO(b))).convert("RGB") for b in subidas.values()]
if imgs:
    fig, ejes = plt.subplots(-(-len(imgs) // 3), min(3, len(imgs)), figsize=(6 * min(3, len(imgs)), 6.5 * -(-len(imgs) // 3)), squeeze=False)
    for ax in ejes.flat: ax.set_visible(False)
    for ax, (nombre, im) in zip(ejes.flat, zip(subidas, imgs)):
        d = ImageDraw.Draw(im); lw = max(3, im.width // 150)
        for x1, y1, x2, y2, pr, psn in predecir(im):
            v = VI.decidir(pr, psn, VI.nitidez_recorte(im, x1, y1, x2, y2), TAU_ROTO, TAU_SANO, NITIDEZ_MIN)
            d.rectangle([x1, y1, x2, y2], outline=COL[v], width=lw)
            d.text((x1 + lw, max(0, y1 - 14 * lw // 3)), f"{ETQ[v]}  roto {pr:.0%}", fill=COL[v], font=ImageFont.load_default(size=6 * lw))
        ax.set_visible(True); ax.imshow(im); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(nombre[:40], fontsize=9)
    plt.tight_layout(); plt.show()

## Integración con la app y el backend

1. Descarga el zip de `MyDrive/eggdetection/` y copia `weights/best.onnx` a `backend/modelos/best.onnx`
   (guarda antes el actual como `best_v1.onnx`). **No hay que tocar código**: la salida `[1, 6, 8400]`
   y el orden de clases `{0: Roto, 1: Sano}` son los mismos del modelo 1 (verificado en la celda 15).
2. En `backend/.env` pon los umbrales que imprimió la celda 17 (`TAU_ROTO`, `TAU_SANO`).
3. YOLOv8l es ~2× más pesado que el modelo 1 en CPU: en tu `t2.large` calcula **~1,3 s por foto**.
   Si la banda necesita respuesta más rápida, reentrena con `MODELO_BASE = "yolov8m.pt"`.

## Fuentes (hay que citarlas en el informe; revisa la licencia de cada una en su página)

`indra-bayu/egg-shell-cracked-detections` · `360digitmg-swnfx/egg-crack-detection-yjhsy` ·
`digit360mg/egg-crack-detection-fahfj` · `new-workspace-jqutx/egg-crack-detection-ek7qj` ·
`carlos-oliveira-5vnxi/crack-egg-8wlgl` · `aau-nupnz/egg-crack-9lbqk` · `crackedeggs/cracked-eggs` ·
`ducks-sfodd/egg-quality-o7xyt` — todas en https://universe.roboflow.com